# Inspect Activation Patching Spans

Manually verify that `character_spans_for_locations` and `token_positions_for_spans` are selecting the correct text/tokens, and that instruction tokens added by the chat template are **not** being patched.

In [25]:
import json
import re
import sys
from pathlib import Path
from IPython.display import display, HTML

_ROOT = Path(".").resolve()
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))
_DOL_INTERP = _ROOT.parents[2]
if str(_DOL_INTERP) not in sys.path:
    sys.path.insert(0, str(_DOL_INTERP))

from boxes_lm_eval import prompt_and_gold, scenario_without_query, build_chat_conversation

## Span-extraction + chat-prompt helpers (verbatim from `run_activation_patching_boxes.py`)

In [26]:
# ── span extraction ──────────────────────────────────────────────────────────

def sentence_spans(text):
    spans = []
    for match in re.finditer(r"[^.]*\.", text):
        start, end = match.span()
        sent = text[start:end].strip()
        if sent:
            spans.append((start, end, sent))
    return spans


def context_statement_spans(text):
    spans = sentence_spans(text)
    if not spans:
        return []
    first_start, first_end, first_sent = spans[0]
    if not re.match(r"\s*Box \d+ contains\b", first_sent):
        return spans
    out = []
    first_text = text[first_start:first_end]
    clause_pattern = re.compile(
        r"Box \d+ contains .*?(?=,\s*Box \d+ contains |\.$)",
        flags=re.DOTALL,
    )
    for match in clause_pattern.finditer(first_text):
        start = first_start + match.start()
        end = first_start + match.end()
        if end < len(text) and text[end] in ",.": end += 1
        clause = text[start:end].strip()
        if clause:
            out.append((start, end, clause))
    if not out:
        out.append((first_start, first_end, first_sent))
    out.extend(spans[1:])
    return out


def character_spans_for_locations(scenario, target_box):
    box_pattern = re.compile(rf"\bBox {target_box}\b")
    target_spans, control_spans = [], []
    for start, end, sent in context_statement_spans(scenario):
        if box_pattern.search(sent):
            target_spans.append((start, end))
        else:
            control_spans.append((start, end))
    return {
        "all_context":              [(0, len(scenario))],
        "target_box_sentences":     target_spans,
        "last_arget_box_sentence":  target_spans[-1:] if target_spans else [],
        "non_target_box_sentences": control_spans,
    }


def token_positions_for_spans(tokenizer, prompt, spans, *, add_special_tokens):
    if not spans:
        return []
    encoded = tokenizer(
        prompt, add_special_tokens=add_special_tokens,
        return_offsets_mapping=True, return_token_type_ids=False,
    )
    offsets = encoded.get("offset_mapping")
    if offsets is None:
        raise ValueError("Tokenizer did not return offset mappings; a fast tokenizer is required")
    positions = []
    for token_idx, offset in enumerate(offsets):
        start, end = int(offset[0]), int(offset[1])
        if start == end:
            continue
        if any(start < span_end and end > span_start for span_start, span_end in spans):
            positions.append(token_idx)
    return positions


def offset_spans(spans, offset):
    return [(s + offset, e + offset) for s, e in spans]


# ── chat prompt helpers ───────────────────────────────────────────────────────

def chat_answers_have_object_prefix(*answers):
    return all(a.strip().startswith("the ") for a in answers)


def add_chat_assistant_object_prefix(conversation):
    out = [dict(m) for m in conversation]
    out[-1]["content"] = out[-1]["content"].rstrip() + " the"
    return out


def render_chat_prompt(tokenizer, conversation):
    try:
        return tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=False,
            continue_final_message=True, enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=False,
            continue_final_message=True,
        )


def post_scenario_box_spans(prompt, *, target_box, scenario_offset, scenario_length):
    """
    Find every 'Box {target_box}' occurrence after the scenario text ends.

    The last occurrence is the assistant-turn response prefix ("Box X contains …").
    All earlier occurrences are user-turn mentions ("What does Box X contain?",
    the example '"Box X contains …"' in the instructions, etc.).

    Returns:
        query_spans:    all occurrences except the last (user-turn mentions)
        response_spans: the last occurrence only (assistant-turn prefix)
    """
    query_start = scenario_offset + scenario_length
    all_spans = [m.span() for m in re.compile(rf"\bBox {target_box}\b").finditer(prompt, pos=query_start)]
    if not all_spans:
        raise ValueError(f"Could not find 'Box {target_box}' after the scenario")
    return all_spans[:-1], all_spans[-1:]

## Config

In [27]:
DATA_FILE    = Path("data/activation_patching_boxes6_1item_nops12/test-t5.jsonl")
MODEL_NAME   = "Qwen/Qwen3-8B"
PROMPT_MODE  = "chat"   # "chat" or "continuation"
N_EXAMPLES   = 3        # rows to inspect inline (Sections 1–3)

# HTML export settings
K_SAMPLES        = 3    # number of skeleton_ids to include
N_VARIANTS       = 2    # variants per skeleton_id
HTML_OUTPUT_FILE = Path("output/span_inspection.html")

# Loading just the tokenizer is fast (no GPU) and enables all token-level views.
# Set LOAD_MODEL=True only if you also want to run the forward pass (not needed here).
LOAD_TOKENIZER = True
LOAD_MODEL     = False

## Load data

In [28]:
rows = [json.loads(l) for l in DATA_FILE.read_text().splitlines() if l.strip()]
print(f"Loaded {len(rows)} rows from {DATA_FILE}")
print("Keys:", list(rows[0].keys()))

Loaded 600 rows from data/activation_patching_boxes6_1item_nops12/test-t5.jsonl
Keys: ['sentence', 'sentence_masked', 'masked_content', 'sample_id', 'skeleton_id', 'variant_id', 'target_box', 'numops', 'numops_by_op', 'object_assignment', 'operation_skeleton', 'token_counts']


## Load tokenizer

In [29]:
tokenizer = None
if LOAD_TOKENIZER:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("Tokenizer loaded:", type(tokenizer).__name__)

if LOAD_MODEL:
    from boxes_lm_eval import load_model_and_tokenizer
    _model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
    print("Model + tokenizer loaded")

Tokenizer loaded: Qwen2Tokenizer


## Helpers: per-row prompt builder

In [30]:
def build_prompt_data(row, tokenizer, prompt_mode):
    """Return everything needed to inspect spans for one row."""
    context, answer = prompt_and_gold(row)
    scenario, target_box = scenario_without_query(context)

    if prompt_mode == "continuation":
        prompt = context
        scenario_offset = 0
        add_special_tokens = True
        post_scenario_spans = {}
    else:  # chat
        conversation, _, _ = build_chat_conversation(row)
        if chat_answers_have_object_prefix(answer):
            conversation = add_chat_assistant_object_prefix(conversation)
        prompt = render_chat_prompt(tokenizer, conversation)
        scenario_offset = prompt.find(scenario)
        if scenario_offset < 0:
            raise ValueError("Could not locate scenario inside rendered chat prompt")
        add_special_tokens = False
        query_spans, response_spans = post_scenario_box_spans(
            prompt, target_box=target_box,
            scenario_offset=scenario_offset, scenario_length=len(scenario),
        )
        post_scenario_spans = {
            "query_box":            query_spans,
            "response_target_box":  response_spans,
        }

    char_locations = character_spans_for_locations(scenario, target_box)
    # Shift scenario-relative character spans to prompt-absolute
    positions_by_location = {
        loc: token_positions_for_spans(
            tokenizer, prompt,
            offset_spans(spans, scenario_offset),
            add_special_tokens=add_special_tokens,
        )
        for loc, spans in char_locations.items()
    }
    # Post-scenario locations (already prompt-absolute)
    for loc, spans in post_scenario_spans.items():
        positions_by_location[loc] = token_positions_for_spans(
            tokenizer, prompt, spans, add_special_tokens=add_special_tokens,
        )
    # last_token: sanity-check; no character span needed
    n_tokens = len(tokenizer(prompt, add_special_tokens=add_special_tokens)["input_ids"])
    positions_by_location["last_token"] = [n_tokens - 1]

    return {
        "prompt": prompt,
        "scenario": scenario,
        "scenario_offset": scenario_offset,
        "scenario_length": len(scenario),
        "target_box": target_box,
        "answer": answer,
        "add_special_tokens": add_special_tokens,
        "char_locations": char_locations,           # scenario-relative character spans
        "post_scenario_spans": post_scenario_spans,  # prompt-absolute character spans
        "positions_by_location": positions_by_location,  # token indices into full prompt
    }

## Rendering helpers

In [31]:
# ── colors ────────────────────────────────────────────────────────────────────

LOCATION_COLORS = {
    "all_context":              "#cce5ff",  # blue
    "target_box_sentences":     "#d4edda",  # green
    "last_arget_box_sentence":  "#fff3cd",  # yellow
    "non_target_box_sentences": "#f8d7da",  # red
    "query_box":                "#e2d9f3",  # purple  — user-turn Box X mentions
    "response_target_box":      "#fde8c8",  # orange  — assistant-turn "Box X contains"
    "last_token":               "#f5c6cb",  # pink    — sanity-check: final token
}

REGION_COLORS = {
    "instruction": "#e9ecef",   # gray  — chat template boilerplate before scenario
    "scenario":    "#ffffff",   # white — scenario tokens not in any patch span
    "query":       "#dde8f0",   # slate — everything after the scenario
}

# Each group is shown as a separate panel to avoid overlapping colors.
LOCATION_GROUPS = [
    ("all context",
     ["all_context"]),
    ("target + non-target",
     ["target_box_sentences", "last_arget_box_sentence", "non_target_box_sentences"]),
    ("last_target + query_box + response_target_box + last_token",
     ["last_arget_box_sentence", "query_box", "response_target_box", "last_token"]),
]


# ── low-level helpers ─────────────────────────────────────────────────────────

def _escape(text):
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _coverage_map(text_len, spans_by_location):
    cov = {}
    for loc, spans in spans_by_location.items():
        for s, e in spans:
            for i in range(s, min(e, text_len)):
                cov[i] = loc
    return cov


def _loc_legend(active_locs):
    """Legend items for the given location names (only those with a known color)."""
    return "".join(
        f'<span style="background:{LOCATION_COLORS[l]};padding:2px 7px;border-radius:3px;'
        f'margin-right:5px;font-size:0.82em">{l}</span>'
        for l in active_locs if l in LOCATION_COLORS
    )


def _region_legend():
    return "".join(
        f'<span style="background:{c};padding:2px 7px;border-radius:3px;margin-right:5px;font-size:0.82em">{l}</span>'
        for l, c in REGION_COLORS.items()
    )


# ── character view (scenario-relative) ───────────────────────────────────────

def highlight_char_spans(text, spans_by_location, title=""):
    """Highlight character spans within the scenario text."""
    cov = _coverage_map(len(text), spans_by_location)
    parts = []
    i = 0
    while i < len(text):
        loc = cov.get(i)
        j = i + 1
        while j < len(text) and cov.get(j) == loc:
            j += 1
        chunk = _escape(text[i:j])
        if loc:
            parts.append(f'<mark style="background:{LOCATION_COLORS.get(loc,"#eee")};border-radius:3px" title="{loc}">{chunk}</mark>')
        else:
            parts.append(chunk)
        i = j
    t = f"<b>{_escape(title)}</b><br>" if title else ""
    return (
        f'<div style="border:1px solid #ccc;border-radius:6px;padding:12px;margin-bottom:6px;'
        f'font-family:monospace;white-space:pre-wrap">{t}'
        f'<b>Patching:</b>&nbsp;{_loc_legend(spans_by_location.keys())}<br><br>'
        f'{"".join(parts)}</div>'
    )


# ── full-prompt character view ────────────────────────────────────────────────

def highlight_full_prompt(prompt, scenario_offset, scenario_length,
                          char_locations, post_scenario_spans, title=""):
    """
    Highlight the full rendered prompt with region + patching-location colors.
    `char_locations` and `post_scenario_spans` should be pre-filtered to the
    locations you want to display.
    """
    scenario_end = scenario_offset + scenario_length
    abs_locs = {loc: offset_spans(spans, scenario_offset) for loc, spans in char_locations.items()}
    abs_locs.update(post_scenario_spans)
    cov = _coverage_map(len(prompt), abs_locs)

    def region_of(i):
        if i < scenario_offset: return "instruction"
        if i >= scenario_end:   return "query"
        return None

    parts = []
    i = 0
    while i < len(prompt):
        loc = cov.get(i)
        reg = region_of(i)
        j = i + 1
        while j < len(prompt) and cov.get(j) == loc and region_of(j) == reg:
            j += 1
        chunk = _escape(prompt[i:j]).replace("\n", "↵\n")
        if loc:
            parts.append(f'<mark style="background:{LOCATION_COLORS.get(loc,"#eee")};border-radius:2px" title="patch:{loc}">{chunk}</mark>')
        elif reg:
            parts.append(f'<span style="background:{REGION_COLORS[reg]}" title="region:{reg}">{chunk}</span>')
        else:
            parts.append(f'<span style="background:{REGION_COLORS["scenario"]}">{chunk}</span>')
        i = j

    t = f"<b>{_escape(title)}</b><br>" if title else ""
    return (
        f'<div style="border:1px solid #ccc;border-radius:6px;padding:12px;margin-bottom:6px;'
        f'font-family:monospace;white-space:pre-wrap">{t}'
        f'<b>Patching:</b>&nbsp;{_loc_legend(abs_locs.keys())}'
        f'&nbsp;&nbsp;<b>Region:</b>&nbsp;{_region_legend()}<br><br>'
        f'{"".join(parts)}</div>'
    )


# ── token view ────────────────────────────────────────────────────────────────

def render_token_view(tokenizer, prompt, scenario_offset, scenario_length,
                      positions_by_location, title="", add_special_tokens=False):
    """
    Render every token with background = patching location and outline = prompt region.
    `positions_by_location` should be pre-filtered to the locations you want to display.
    """
    encoded = tokenizer(prompt, add_special_tokens=add_special_tokens,
                        return_offsets_mapping=True, return_token_type_ids=False)
    ids     = encoded["input_ids"]
    offsets = encoded["offset_mapping"]
    tokens  = tokenizer.convert_ids_to_tokens(ids)
    scenario_end = scenario_offset + scenario_length

    pos_to_loc = {p: loc for loc, positions in positions_by_location.items() for p in positions}
    BORDER = {"instruction": "#adb5bd", "scenario": "#6c757d", "query": "#6aa3c8"}

    parts = []
    for idx, (tok, offset) in enumerate(zip(tokens, offsets)):
        cs, ce = int(offset[0]), int(offset[1])
        dt = (tok.replace("Ġ", "▁").replace("Ċ", "↵")
                 .replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))
        if ce <= scenario_offset or (cs == ce and cs <= scenario_offset): region = "instruction"
        elif cs >= scenario_end:                                           region = "query"
        else:                                                              region = "scenario"
        bc  = BORDER[region]
        tip = f"pos={idx} | chars={cs}:{ce} | region={region}"
        if idx in pos_to_loc:
            loc  = pos_to_loc[idx]
            tip += f" | patch={loc}"
            parts.append(
                f'<mark style="background:{LOCATION_COLORS.get(loc,"#eee")};outline:2px solid {bc};'
                f'border-radius:3px;margin:1px;padding:1px 3px;" title="{tip}">{dt}</mark>'
            )
        else:
            parts.append(
                f'<span style="background:{REGION_COLORS.get(region,"#fff")};outline:1px solid {bc};'
                f'border-radius:2px;margin:1px;padding:1px 2px;" title="{tip}">{dt}</span>'
            )

    t = f"<b>{_escape(title)}</b><br>" if title else ""
    return (
        f'<div style="border:1px solid #ccc;border-radius:6px;padding:12px;margin-bottom:14px;'
        f'font-family:monospace;line-height:2.2">{t}'
        f'<b>Patched (colored bg):</b>&nbsp;{_loc_legend(positions_by_location.keys())}'
        f'&nbsp;&nbsp;<b>Region (border):</b>&nbsp;{_region_legend()}<br><br>'
        f'{" ".join(parts)}</div>'
    )


# ── grouped sample rendering ──────────────────────────────────────────────────

def render_sample_html(pd, tokenizer, title=""):
    """
    Render one sample split into one panel per LOCATION_GROUP.
    Each group gets its own full-prompt char view and token view so colors don't overlap.
    """
    parts = []
    if title:
        parts.append(
            f'<div style="font-family:sans-serif;font-weight:bold;font-size:1.05em;'
            f'margin-bottom:10px">{_escape(title)}</div>'
        )
    for group_label, locs in LOCATION_GROUPS:
        loc_set = set(locs)
        f_char = {k: v for k, v in pd["char_locations"].items()        if k in loc_set}
        f_post = {k: v for k, v in pd["post_scenario_spans"].items()   if k in loc_set}
        f_pos  = {k: v for k, v in pd["positions_by_location"].items() if k in loc_set}
        parts.append(
            f'<div style="font-family:sans-serif;color:#333;font-size:0.92em;'
            f'margin:14px 0 3px;padding:4px 8px;background:#f5f5f5;'
            f'border-left:3px solid #888;border-radius:2px">'
            f'<b>{_escape(group_label)}</b></div>'
        )
        parts.append(highlight_full_prompt(
            pd["prompt"], pd["scenario_offset"], pd["scenario_length"], f_char, f_post,
        ))
        parts.append(render_token_view(
            tokenizer, pd["prompt"], pd["scenario_offset"], pd["scenario_length"],
            f_pos, add_special_tokens=pd["add_special_tokens"],
        ))
    return "".join(parts)


def build_html_page(body_html, title="Activation Patching Span Inspection"):
    return f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>{title}</title>
<style>
  body {{ font-family:sans-serif; max-width:1600px; margin:0 auto; padding:24px; background:#f7f7f7; }}
  h1, h2 {{ color:#222; }}
  .skeleton-block {{
    background:#fff; border:1px solid #ddd; border-radius:8px;
    padding:20px; margin-bottom:36px; box-shadow:0 1px 4px rgba(0,0,0,.06);
  }}
  .variant-pair {{ display:flex; gap:20px; }}
  .variant {{
    flex:1; min-width:0; border:1px solid #e0e0e0; border-radius:6px;
    padding:14px; background:#fdfdfd;
  }}
  .variant-hdr {{ font-size:0.9em; color:#555; margin-bottom:10px; font-family:monospace; }}
</style>
</head>
<body>
<h1>{title}</h1>
{body_html}
</body>
</html>
"""

---
## Section 1 — Character spans within the scenario (no tokenizer needed)

Quickly verify that the regex-based span extraction picks the right sentences/clauses.

In [32]:
for row in rows[:N_EXAMPLES]:
    context, answer = prompt_and_gold(row)
    scenario, target_box = scenario_without_query(context)
    locations = character_spans_for_locations(scenario, target_box)

    hdr = f"skeleton_id={row.get('skeleton_id')}  variant_id={row.get('variant_id')}  target_box={target_box}  answer='{answer}'"
    print(hdr)
    for loc, spans in locations.items():
        print(f"  [{loc}] → {[scenario[s:e] for s, e in spans]}")
    print()
    display(HTML(highlight_char_spans(scenario, locations, title=hdr)))

skeleton_id=0  variant_id=0  target_box=4  answer='the fan'
  [all_context] → ['Box 0 contains the tea, Box 1 contains the file, Box 2 contains the pot, Box 3 contains the flower, Box 4 contains the fan, Box 5 contains the tissue. Remove the flower from Box 3. Put the brain into Box 3. Remove the tea from Box 0. Move the tissue from Box 5 to Box 0. Move the pot from Box 2 to Box 5. Move the file from Box 1 to Box 2. Move the tissue from Box 0 to Box 1. Put the shell into Box 0. Remove the file from Box 2. Move the shell from Box 0 to Box 2. Move the tissue from Box 1 to Box 0. Move the pot from Box 5 to Box 1.']
  [target_box_sentences] → ['Box 4 contains the fan,']
  [last_arget_box_sentence] → ['Box 4 contains the fan,']
  [non_target_box_sentences] → ['Box 0 contains the tea,', 'Box 1 contains the file,', 'Box 2 contains the pot,', 'Box 3 contains the flower,', 'Box 5 contains the tissue.', ' Remove the flower from Box 3.', ' Put the brain into Box 3.', ' Remove the tea from Box 0.'

skeleton_id=0  variant_id=1  target_box=4  answer='the phone'
  [all_context] → ['Box 0 contains the tape, Box 1 contains the chemical, Box 2 contains the rock, Box 3 contains the disk, Box 4 contains the phone, Box 5 contains the key. Remove the disk from Box 3. Put the leaf into Box 3. Remove the tape from Box 0. Move the key from Box 5 to Box 0. Move the rock from Box 2 to Box 5. Move the chemical from Box 1 to Box 2. Move the key from Box 0 to Box 1. Put the camera into Box 0. Remove the chemical from Box 2. Move the camera from Box 0 to Box 2. Move the key from Box 1 to Box 0. Move the rock from Box 5 to Box 1.']
  [target_box_sentences] → ['Box 4 contains the phone,']
  [last_arget_box_sentence] → ['Box 4 contains the phone,']
  [non_target_box_sentences] → ['Box 0 contains the tape,', 'Box 1 contains the chemical,', 'Box 2 contains the rock,', 'Box 3 contains the disk,', 'Box 5 contains the key.', ' Remove the disk from Box 3.', ' Put the leaf into Box 3.', ' Remove the tape fro

skeleton_id=0  variant_id=2  target_box=4  answer='the milk'
  [all_context] → ['Box 0 contains the letter, Box 1 contains the apple, Box 2 contains the plane, Box 3 contains the block, Box 4 contains the milk, Box 5 contains the medicine. Remove the block from Box 3. Put the fish into Box 3. Remove the letter from Box 0. Move the medicine from Box 5 to Box 0. Move the plane from Box 2 to Box 5. Move the apple from Box 1 to Box 2. Move the medicine from Box 0 to Box 1. Put the cross into Box 0. Remove the apple from Box 2. Move the cross from Box 0 to Box 2. Move the medicine from Box 1 to Box 0. Move the plane from Box 5 to Box 1.']
  [target_box_sentences] → ['Box 4 contains the milk,']
  [last_arget_box_sentence] → ['Box 4 contains the milk,']
  [non_target_box_sentences] → ['Box 0 contains the letter,', 'Box 1 contains the apple,', 'Box 2 contains the plane,', 'Box 3 contains the block,', 'Box 5 contains the medicine.', ' Remove the block from Box 3.', ' Put the fish into Box 3.', 

## Section 2 — `context_statement_spans` detail (clause splitting of initial inventory)

In [33]:
row = rows[0]
context, _ = prompt_and_gold(row)
scenario, target_box = scenario_without_query(context)
print(f"target_box={target_box}\n")
for i, (start, end, sent) in enumerate(context_statement_spans(scenario)):
    print(f"  [{i:02d}] chars [{start:3d}:{end:3d}] → {sent!r}")

target_box=4

  [00] chars [  0: 23] → 'Box 0 contains the tea,'
  [01] chars [ 24: 48] → 'Box 1 contains the file,'
  [02] chars [ 49: 72] → 'Box 2 contains the pot,'
  [03] chars [ 73: 99] → 'Box 3 contains the flower,'
  [04] chars [100:123] → 'Box 4 contains the fan,'
  [05] chars [124:150] → 'Box 5 contains the tissue.'
  [06] chars [150:180] → 'Remove the flower from Box 3.'
  [07] chars [180:206] → 'Put the brain into Box 3.'
  [08] chars [206:233] → 'Remove the tea from Box 0.'
  [09] chars [233:270] → 'Move the tissue from Box 5 to Box 0.'
  [10] chars [270:304] → 'Move the pot from Box 2 to Box 5.'
  [11] chars [304:339] → 'Move the file from Box 1 to Box 2.'
  [12] chars [339:376] → 'Move the tissue from Box 0 to Box 1.'
  [13] chars [376:402] → 'Put the shell into Box 0.'
  [14] chars [402:430] → 'Remove the file from Box 2.'
  [15] chars [430:466] → 'Move the shell from Box 0 to Box 2.'
  [16] chars [466:503] → 'Move the tissue from Box 1 to Box 0.'
  [17] chars [503:537] 

---
## Section 3 — Grouped views *(requires tokenizer)*

Each sample is split into **three panels** — one per `LOCATION_GROUP` — so colors never overlap:

1. **all context** — the entire scenario highlighted blue
2. **target + non-target** — green/yellow for target-box sentences, red for non-target
3. **last\_target + query\_box + response\_target\_box + last\_token** — fine-grained post-context locations

Each panel shows the full rendered prompt (character-level) and the token-level view.  
Instruction-prefix tokens are gray, query/assistant tokens are slate — neither should appear colored.

In [34]:
if tokenizer is None:
    print("Set LOAD_TOKENIZER=True in the Config cell.")
else:
    for row in rows[:N_EXAMPLES]:
        pd = build_prompt_data(row, tokenizer, PROMPT_MODE)
        hdr = (
            f"skeleton_id={row.get('skeleton_id')}  variant_id={row.get('variant_id')}  "
            f"target_box={pd['target_box']}  answer='{pd['answer']}'"
        )
        display(HTML(render_sample_html(pd, tokenizer, title=hdr)))

---
## Section 4 — Export to HTML *(requires tokenizer)*

Writes `K_SAMPLES` skeleton IDs × `N_VARIANTS` variants to a self-contained HTML file.  
Each skeleton gets a side-by-side two-column layout (one column per variant) with all three group panels inside each column.

In [35]:
if tokenizer is None:
    print("Set LOAD_TOKENIZER=True in the Config cell.")
else:
    from collections import defaultdict

    by_skeleton = defaultdict(list)
    for row in rows:
        by_skeleton[row["skeleton_id"]].append(row)

    blocks = []
    for skeleton_id in sorted(by_skeleton)[:K_SAMPLES]:
        variants = sorted(by_skeleton[skeleton_id], key=lambda r: int(r["variant_id"]))[:N_VARIANTS]

        pair_cols = []
        for v_row in variants:
            pd = build_prompt_data(v_row, tokenizer, PROMPT_MODE)
            hdr = (
                f"variant_id={v_row['variant_id']}  "
                f"target_box={pd['target_box']}  answer='{pd['answer']}'"
            )
            pair_cols.append(
                f'<div class="variant">'
                f'<div class="variant-hdr">{_escape(hdr)}</div>'
                f'{render_sample_html(pd, tokenizer)}'
                f'</div>'
            )

        blocks.append(
            f'<div class="skeleton-block">'
            f'<h2 style="margin-top:0">skeleton_id = {skeleton_id}</h2>'
            f'<div class="variant-pair">{"".join(pair_cols)}</div>'
            f'</div>'
        )

    html = build_html_page("".join(blocks))
    HTML_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    HTML_OUTPUT_FILE.write_text(html, encoding="utf-8")
    print(f"Wrote {K_SAMPLES} skeletons × {N_VARIANTS} variants → {HTML_OUTPUT_FILE}")
    display(HTML(f'<a href="{HTML_OUTPUT_FILE.resolve()}" target="_blank">Open {HTML_OUTPUT_FILE.name}</a>'))

Wrote 3 skeletons × 2 variants → output/span_inspection.html


---
## Section 5 — Spot-check a specific skeleton_id / variant

In [36]:
SKELETON_ID = 0
VARIANT_ID  = 0

match = next(
    (r for r in rows
     if r.get("skeleton_id") == SKELETON_ID and int(r.get("variant_id", -1)) == VARIANT_ID),
    None,
)
if match is None:
    print(f"No row for skeleton_id={SKELETON_ID} variant_id={VARIANT_ID}")
else:
    context, answer = prompt_and_gold(match)
    scenario, target_box = scenario_without_query(context)
    print(f"Scenario (target_box={target_box}, answer='{answer}'):")
    print(scenario)
    print()

    # Character-level span listing
    locations = character_spans_for_locations(scenario, target_box)
    for loc, spans in locations.items():
        print(f"[{loc}]")
        for s, e in spans:
            print(f"  chars [{s}:{e}]  {scenario[s:e]!r}")

    # Character-level grouped highlight (scenario only, no tokenizer needed)
    hdr = f"skeleton_id={SKELETON_ID}  variant_id={VARIANT_ID}  target_box={target_box}"
    for group_label, locs in LOCATION_GROUPS:
        loc_set = set(locs)
        f_char = {k: v for k, v in locations.items() if k in loc_set}
        if f_char:
            display(HTML(highlight_char_spans(scenario, f_char, title=f"{group_label}")))

    if tokenizer is not None:
        pd = build_prompt_data(match, tokenizer, PROMPT_MODE)
        print("\nPost-scenario spans (prompt-absolute):")
        for loc, spans in pd["post_scenario_spans"].items():
            print(f"  [{loc}] → {[pd['prompt'][s:e] for s, e in spans]}")
        print(f"  [last_token] → pos={pd['positions_by_location']['last_token']}")
        print()
        display(HTML(render_sample_html(pd, tokenizer, title=hdr + "  (full grouped view)")))

Scenario (target_box=4, answer='the fan'):
Box 0 contains the tea, Box 1 contains the file, Box 2 contains the pot, Box 3 contains the flower, Box 4 contains the fan, Box 5 contains the tissue. Remove the flower from Box 3. Put the brain into Box 3. Remove the tea from Box 0. Move the tissue from Box 5 to Box 0. Move the pot from Box 2 to Box 5. Move the file from Box 1 to Box 2. Move the tissue from Box 0 to Box 1. Put the shell into Box 0. Remove the file from Box 2. Move the shell from Box 0 to Box 2. Move the tissue from Box 1 to Box 0. Move the pot from Box 5 to Box 1.

[all_context]
  chars [0:537]  'Box 0 contains the tea, Box 1 contains the file, Box 2 contains the pot, Box 3 contains the flower, Box 4 contains the fan, Box 5 contains the tissue. Remove the flower from Box 3. Put the brain into Box 3. Remove the tea from Box 0. Move the tissue from Box 5 to Box 0. Move the pot from Box 2 to Box 5. Move the file from Box 1 to Box 2. Move the tissue from Box 0 to Box 1. Put the s


Post-scenario spans (prompt-absolute):
  [query_box] → ['Box 4', 'Box 4']
  [response_target_box] → ['Box 4']
  [last_token] → pos=[245]

